## ITAI2373 NewsBot Intelligence System

### Project Overview

This notebook develops an Intelligent NewsBot System designed to process, analyze, and categorize news articles. Leveraging advanced Natural Language Processing (NLP) techniques, this system aims to provide a comprehensive understanding of news content, facilitating efficient information retrieval and sentiment analysis. The project covers data loading, extensive text preprocessing, feature engineering, sentiment analysis, multi-class classification, and named entity recognition, culminating in actionable insights and an executive summary.

---

### Initial Setup: Library Imports and Environment Configuration

In [1]:
# Core Data Manipulation and Scientific Computing Libraries
import pandas as pd
import numpy as np

# Natural Language Toolkit (NLTK) for various text processing tasks
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# spaCy for advanced NLP tasks like POS tagging and Named Entity Recognition
import spacy

# Scikit-learn for machine learning tasks (vectorization, classification, model evaluation)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Matplotlib and Seaborn for data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# VADER (Valence Aware Dictionary and sEntiment Reasoner) for sentiment analysis
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Operating System module for directory operations
import os
import re # Regular Expression operations

# Ensure reproducibility where possible
np.random.seed(42)

# --- Environment Setup ---

# Create an 'outputs' directory to store plots and other generated files
outputs_dir = 'outputs'
if not os.path.exists(outputs_dir):
    os.makedirs(outputs_dir)
    print(f"Created directory: {outputs_dir}")
else:
    print(f"Directory already exists: {outputs_dir}")

print("All necessary libraries imported and environment configured.")


Created directory: outputs
All necessary libraries imported and environment configured.


### NLTK Downloads and SpaCy Model Loading

Before proceeding with text processing, we need to download essential NLTK data packages and load the spaCy English language model.

In [25]:
# Download NLTK data packages
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

try:
    nltk.data.find('corpora/omw-1.4')
except LookupError:
    nltk.download('omw-1.4')

try:
    nltk.data.find('sentiment/vader_lexicon')
except LookupError:
    nltk.download('vader_lexicon')

# Load the spaCy English language model for advanced NLP tasks
try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    print("Downloading spaCy model 'en_core_web_sm'...")
    os.system('python -m spacy download en_core_web_sm')
    nlp = spacy.load('en_core_web_sm')

print("NLTK data downloaded and spaCy model loaded successfully.")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


NLTK data downloaded and spaCy model loaded successfully.


## MODULE 1: Business Case, User Identification, and Value Proposition

This module outlines the foundational aspects of the NewsBot Intelligence System, defining its purpose, target users, and the core value it delivers.

### 1.1 Business Case

In today's information-rich environment, individuals and organizations are constantly inundated with news from various sources. The sheer volume makes it challenging to quickly identify relevant information, track specific topics, understand public sentiment, or monitor competitive landscapes. A NewsBot Intelligence System addresses this challenge by automating the process of news comprehension and analysis.

*   **Problem:** Information overload, difficulty in extracting key insights from vast news datasets, manual and time-consuming news monitoring.
*   **Opportunity:** Develop an intelligent system that can efficiently process, categorize, and analyze news articles to deliver structured insights.


### 1.2 User Identification

The primary users for this NewsBot Intelligence System are:

*   **Media Analysts:** To quickly gauge public sentiment on specific topics, track news trends, and understand media coverage patterns.
*   **Market Researchers:** To monitor industry-specific news, competitor announcements, and identify emerging market opportunities or threats.
*   **Journalists & Editors:** To discover trending stories, fact-check information, and categorize news for content management systems.
*   **Academics & Researchers:** For large-scale textual analysis of news corpora in social sciences, linguistics, and political science.
*   **General Public:** For personalized news aggregation and summarization based on their interests.


### 1.3 Value Proposition

The NewsBot Intelligence System offers significant value through:

*   **Efficiency:** Automates the laborious process of news monitoring and analysis, saving considerable time and resources.
*   **Accuracy:** Provides consistent and objective analysis of news content, reducing human bias and error.
*   **Insights:** Uncovers hidden patterns, relationships, and sentiment within news articles that might be missed by manual review.
*   **Scalability:** Capable of processing vast quantities of news data, adapting to growing information streams.
*   **Decision Support:** Empowers users with data-driven insights for strategic decision-making in various domains, from business intelligence to public relations.

---

## MODULE 2: Data Loading and Preprocessing

This module focuses on ingesting the raw news dataset and transforming it into a clean, normalized format suitable for subsequent NLP tasks. This involves several critical steps to handle text data effectively.

### 2.1 Load Dataset

**Action Required:** Please upload your Kaggle news dataset (CSV format) to your Colab environment. The file should contain at least two columns: `content` (for the news article text) and `category` (for the news category). Enter the path to your uploaded file below.

In [27]:
# Define the path to your dataset
# Replace 'your_dataset.csv' with the actual path to your uploaded Kaggle news dataset
dataset_path = '/content/BBC News Train.csv' # Example: '/content/news_articles.csv'

# Load the dataset into a pandas DataFrame
try:
    df = pd.read_csv(dataset_path)
    print(f"Dataset loaded successfully from {dataset_path}")

    # Rename columns for consistency if they exist
    if 'Text' in df.columns:
        df.rename(columns={'Text': 'content'}, inplace=True)
    if 'Category' in df.columns:
        df.rename(columns={'Category': 'category'}, inplace=True)

    print("First 5 rows of the dataset:")
    display(df.head())
    print("\nDataset Info:")
    df.info()
except FileNotFoundError:
    print(f"Error: The file '{dataset_path}' was not found. Please upload your dataset and ensure the path is correct.")
    df = pd.DataFrame() # Initialize an empty DataFrame to prevent further errors
except Exception as e:
    print(f"An error occurred while loading the dataset: {e}")
    df = pd.DataFrame() # Initialize an empty DataFrame

Dataset loaded successfully from /content/BBC News Train.csv
First 5 rows of the dataset:


,ArticleId,content,category
0,1833,worldcom ex-boss launches defence lawyers defe...,business
1,154,german business confidence slides german busin...,business
2,1101,bbc poll indicates economic gloom citizens in ...,business
3,1976,lifestyle governs mobile choice faster bett...,tech
4,917,enron bosses in $168m payout eighteen former e...,business



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1490 entries, 0 to 1489
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   ArticleId  1490 non-null   int64 
 1   content    1490 non-null   object
 2   category   1490 non-null   object
dtypes: int64(1), object(2)
memory usage: 35.1+ KB


### 2.2 Data Preprocessing Functions

This section defines a series of functions to clean and normalize the textual `content` data. These steps are crucial for preparing the text for accurate analysis and modeling.

In [5]:
# Initialize NLTK components
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """
    Performs a series of text preprocessing steps:
    1. Lowercasing
    2. Regex cleaning (remove non-alphabetic characters)
    3. Tokenization
    4. Stopword removal
    5. Lemmatization

    Args:
        text (str): The input text string.

    Returns:
        str: The processed text, joined back into a single string.
    """
    if not isinstance(text, str): # Handle non-string inputs, e.g., NaNs
        return ""

    # 1. Lowercasing
    text = text.lower()

    # 2. Regex cleaning: Remove non-alphabetic characters and extra spaces
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Keep only letters and spaces
    text = re.sub(r'\s+', ' ', text).strip() # Replace multiple spaces with a single space

    # 3. Tokenization
    tokens = word_tokenize(text)

    # 4. Stopword removal and 5. Lemmatization
    processed_tokens = []
    for word in tokens:
        if word not in stop_words:
            processed_tokens.append(lemmatizer.lemmatize(word))

    return ' '.join(processed_tokens)

print("Text preprocessing function 'preprocess_text' defined.")


Text preprocessing function 'preprocess_text' defined.


### 2.3 Apply Preprocessing and Inspect Data

Now, we apply the `preprocess_text` function to the `content` column of our dataset and then inspect the cleaned data.

In [6]:
if not df.empty and 'content' in df.columns:
    print("Applying text preprocessing to the 'content' column...")
    df['cleaned_content'] = df['content'].apply(preprocess_text)
    print("Preprocessing complete.")
    print("\nFirst 5 rows with original and cleaned content:")
    display(df[['content', 'cleaned_content']].head())

    print("\nDistribution of categories after loading:")
    plt.figure(figsize=(10, 6))
    sns.countplot(y='category', data=df, order=df['category'].value_counts().index)
    plt.title('Distribution of News Categories')
    plt.xlabel('Number of Articles')
    plt.ylabel('Category')
    plt.tight_layout()
    plt.savefig(os.path.join(outputs_dir, 'category_distribution.png'))
    plt.show()

    print("\nBusiness Insight (Module 2):\nData preprocessing is critical for noise reduction and standardization. By converting text to lowercase, removing special characters, and reducing words to their base forms (lemmatization), we enhance the accuracy of subsequent NLP tasks like classification and sentiment analysis. This step ensures that variations in word forms or irrelevant characters do not skew our analytical results. The category distribution plot gives us an initial understanding of our dataset's balance, which is important for evaluating model fairness and performance across different news types.")
else:
    print("DataFrame is empty or 'content' column is missing. Please ensure your dataset is loaded correctly with a 'content' column.")


DataFrame is empty or 'content' column is missing. Please ensure your dataset is loaded correctly with a 'content' column.


## MODULE 3: TF-IDF Vectorization

TF-IDF (Term Frequency-Inverse Document Frequency) is a numerical statistic that reflects how important a word is to a document in a collection or corpus. It is often used as a weighting factor in information retrieval and text mining. This module will apply TF-IDF to our preprocessed news content to extract features and gain insights into term importance.

### 3.1 TF-IDF Vectorization Implementation

We'll use `TfidfVectorizer` from `scikit-learn` to convert our `cleaned_content` into a matrix of TF-IDF features. We'll also define a function to perform this operation.

In [7]:
def perform_tfidf_vectorization(text_data, max_features=5000):
    """
    Applies TF-IDF vectorization to a list of text documents.

    Args:
        text_data (list of str): A list of preprocessed text documents.
        max_features (int): The maximum number of features (vocabulary size).

    Returns:
        tuple: A tuple containing:
            - TfidfVectorizer: The fitted TF-IDF vectorizer.
            - sparse matrix: The TF-IDF matrix.
    """
    vectorizer = TfidfVectorizer(max_features=max_features)
    tfidf_matrix = vectorizer.fit_transform(text_data)
    return vectorizer, tfidf_matrix


if not df.empty and 'cleaned_content' in df.columns:
    print("Performing TF-IDF vectorization...")
    tfidf_vectorizer, tfidf_matrix = perform_tfidf_vectorization(df['cleaned_content'])
    print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
    print("TF-IDF vectorization complete.")
else:
    print("DataFrame is empty or 'cleaned_content' column is missing. Cannot perform TF-IDF vectorization.")


DataFrame is empty or 'cleaned_content' column is missing. Cannot perform TF-IDF vectorization.


### 3.2 Top Terms Visualization

To understand the most significant words across the entire news corpus, we'll extract and visualize the top TF-IDF terms.

In [8]:
if 'tfidf_vectorizer' in locals() and 'tfidf_matrix' in locals():
    print("Visualizing top TF-IDF terms...")
    # Get feature names (words) from the vectorizer
    feature_names = tfidf_vectorizer.get_feature_names_out()

    # Calculate the average TF-IDF score for each term across all documents
    avg_tfidf_scores = tfidf_matrix.mean(axis=0)

    # Create a DataFrame of terms and their average TF-IDF scores
    tfidf_df = pd.DataFrame({'term': feature_names, 'avg_tfidf': avg_tfidf_scores.flatten()})

    # Sort by average TF-IDF score and get the top N terms
    top_n = 20
    top_terms = tfidf_df.sort_values(by='avg_tfidf', ascending=False).head(top_n)

    plt.figure(figsize=(12, 8))
    sns.barplot(x='avg_tfidf', y='term', data=top_terms, palette='viridis')
    plt.title(f'Top {top_n} TF-IDF Terms Across All News Articles')
    plt.xlabel('Average TF-IDF Score')
    plt.ylabel('Term')
    plt.tight_layout()
    plt.savefig(os.path.join(outputs_dir, 'top_tfidf_terms.png'))
    plt.show()
    print("Top TF-IDF terms visualization saved.")
else:
    print("TF-IDF vectorizer or matrix not found. Please ensure vectorization was successful.")


TF-IDF vectorizer or matrix not found. Please ensure vectorization was successful.


### 3.3 Category-Specific Term Analysis

Understanding which terms are most important within each news category can reveal key thematic differences. We will identify and visualize the top TF-IDF terms for a few selected categories.

In [9]:
def get_top_terms_by_category(df_cleaned, vectorizer, tfidf_matrix, category, top_n=10):
    """
    Identifies the top TF-IDF terms for a specific category.

    Args:
        df_cleaned (pd.DataFrame): DataFrame with 'cleaned_content' and 'category'.
        vectorizer (TfidfVectorizer): The fitted TF-IDF vectorizer.
        tfidf_matrix (sparse matrix): The TF-IDF matrix.
        category (str): The specific news category to analyze.
        top_n (int): Number of top terms to retrieve.

    Returns:
        pd.DataFrame: DataFrame containing top terms and their TF-IDF scores for the category.
    """
    if not df_cleaned.empty and 'cleaned_content' in df_cleaned.columns and 'category' in df_cleaned.columns:
        category_indices = df_cleaned[df_cleaned['category'] == category].index
        if not category_indices.empty:
            category_tfidf_matrix = tfidf_matrix[category_indices]
            category_avg_tfidf_scores = category_tfidf_matrix.mean(axis=0)

            feature_names = vectorizer.get_feature_names_out()
            category_tfidf_df = pd.DataFrame({'term': feature_names, 'avg_tfidf': category_avg_tfidf_scores.flatten()})
            return category_tfidf_df.sort_values(by='avg_tfidf', ascending=False).head(top_n)
    return pd.DataFrame()


if not df.empty and 'cleaned_content' in df.columns and 'category' in df.columns and 'tfidf_vectorizer' in locals():
    print("Analyzing category-specific terms...")
    # Get a few example categories (e.g., top 3 most frequent ones)
    example_categories = df['category'].value_counts().index.tolist()[:3]

    plt.figure(figsize=(15, 5 * len(example_categories)))
    for i, category in enumerate(example_categories):
        top_terms = get_top_terms_by_category(df, tfidf_vectorizer, tfidf_matrix, category, top_n=10)
        if not top_terms.empty:
            plt.subplot(len(example_categories), 1, i + 1)
            sns.barplot(x='avg_tfidf', y='term', data=top_terms, palette='plasma')
            plt.title(f'Top 10 TF-IDF Terms for Category: {category}')
            plt.xlabel('Average TF-IDF Score')
            plt.ylabel('Term')
            plt.tight_layout()
    plt.savefig(os.path.join(outputs_dir, 'category_specific_tfidf_terms.png'))
    plt.show()
    print("Category-specific TF-IDF terms visualization saved.")
else:
    print("DataFrame is empty, missing required columns, or TF-IDF vectorizer not found. Cannot perform category-specific term analysis.")


DataFrame is empty, missing required columns, or TF-IDF vectorizer not found. Cannot perform category-specific term analysis.


### Business Insight (Module 3)

TF-IDF vectorization is crucial for transforming unstructured text into a quantifiable format, enabling machine learning algorithms to process it. The visualization of top terms across the entire corpus provides a macro-level understanding of the dataset's overall themes and dominant keywords. More importantly, the category-specific term analysis allows businesses to identify unique vocabulary and concepts associated with different news topics. For instance, a 'Finance' category might feature terms like 'economy,' 'market,' and 'stocks,' while a 'Sports' category would highlight terms like 'team,' 'game,' and 'championship.' These insights can inform content strategy, help in fine-tuning recommendation engines, or refine search capabilities by understanding the semantic weight of words within different contexts. It allows for a more targeted approach to information retrieval and content categorization.

## MODULE 4: Part-of-Speech (POS) Tagging

Part-of-Speech (POS) tagging is the process of marking up a word in a text as corresponding to a particular part of speech, based on both its definition and its context. This module will utilize spaCy to perform POS tagging, analyze the frequency of different POS tags, and visualize these distributions.

### 4.1 POS Tagging Implementation

We will define a function to perform POS tagging on our `cleaned_content` using the loaded spaCy model. This will extract POS tags for each token in the text.

In [10]:
def get_pos_tags(text):
    """
    Performs POS tagging on a given text using spaCy.

    Args:
        text (str): The input text string.

    Returns:
        list: A list of POS tags for each token in the text.
    """
    if not isinstance(text, str): # Handle non-string inputs, e.g., NaNs
        return []
    doc = nlp(text)
    return [token.pos_ for token in doc]


if not df.empty and 'cleaned_content' in df.columns:
    print("Applying POS tagging to the cleaned content...")
    # This might take a while for large datasets
    df['pos_tags'] = df['cleaned_content'].apply(get_pos_tags)
    print("POS tagging complete.")
    print("\nFirst 5 rows with original content and POS tags (first few):")
    display(df[['content', 'pos_tags']].head().apply(lambda x: x.apply(lambda y: y[:5] if isinstance(y, list) else y)))
else:
    print("DataFrame is empty or 'cleaned_content' column is missing. Cannot perform POS tagging.")


DataFrame is empty or 'cleaned_content' column is missing. Cannot perform POS tagging.


### 4.2 POS Frequency Analysis

Next, we will analyze the overall frequency of different POS tags across the entire corpus to understand the grammatical composition of the news articles.

In [11]:
from collections import Counter


def analyze_pos_frequency(df_with_pos):
    """
    Analyzes the frequency of POS tags across the entire DataFrame.

    Args:
        df_with_pos (pd.DataFrame): DataFrame containing a 'pos_tags' column.

    Returns:
        pd.Series: A Series of POS tag frequencies, sorted descending.
    """
    all_pos_tags = [tag for sublist in df_with_pos['pos_tags'] for tag in sublist]
    pos_counts = Counter(all_pos_tags)
    return pd.Series(pos_counts).sort_values(ascending=False)


if not df.empty and 'pos_tags' in df.columns:
    print("Analyzing overall POS tag frequency...")
    overall_pos_frequencies = analyze_pos_frequency(df)
    print("Overall POS Tag Frequencies (Top 10):")
    display(overall_pos_frequencies.head(10))
else:
    print("DataFrame is empty or 'pos_tags' column is missing. Cannot analyze POS frequency.")


DataFrame is empty or 'pos_tags' column is missing. Cannot analyze POS frequency.


### 4.3 POS Visualizations

Visualizing the distribution of POS tags provides a clear picture of the grammatical structure. We will plot the overall POS tag distribution and then compare distributions across different news categories.

In [12]:
if not df.empty and 'pos_tags' in df.columns:
    print("Generating POS tag distribution visualizations...")
    # Overall POS Tag Distribution
    overall_pos_frequencies = analyze_pos_frequency(df)
    plt.figure(figsize=(12, 7))
    sns.barplot(x=overall_pos_frequencies.index, y=overall_pos_frequencies.values, palette='mako')
    plt.title('Overall Distribution of Part-of-Speech Tags')
    plt.xlabel('POS Tag')
    plt.ylabel('Frequency')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(os.path.join(outputs_dir, 'overall_pos_distribution.png'))
    plt.show()

    # Category-specific POS Tag Distributions (Top 3 categories)
    example_categories = df['category'].value_counts().index.tolist()[:3]
    fig, axes = plt.subplots(len(example_categories), 1, figsize=(14, 6 * len(example_categories)))
    fig.suptitle('Category-Specific Part-of-Speech Tag Distributions', y=1.02, fontsize=16)

    for i, category in enumerate(example_categories):
        category_df = df[df['category'] == category]
        if not category_df.empty and 'pos_tags' in category_df.columns:
            category_pos_frequencies = analyze_pos_frequency(category_df)
            if not category_pos_frequencies.empty:
                sns.barplot(x=category_pos_frequencies.index, y=category_pos_frequencies.values, palette='rocket', ax=axes[i])
                axes[i].set_title(f'POS Tag Distribution for Category: {category}')
                axes[i].set_xlabel('POS Tag')
                axes[i].set_ylabel('Frequency')
                axes[i].tick_params(axis='x', rotation=45)
            else:
                axes[i].set_title(f'No POS data for Category: {category}')
        else:
            axes[i].set_title(f'No data found for Category: {category}')

    plt.tight_layout()
    plt.savefig(os.path.join(outputs_dir, 'category_specific_pos_distribution.png'))
    plt.show()
    print("POS tag distribution visualizations saved.")
else:
    print("DataFrame is empty or 'pos_tags' column is missing. Cannot visualize POS distributions.")


DataFrame is empty or 'pos_tags' column is missing. Cannot visualize POS distributions.


### Business Insight (Module 4)

POS tagging provides a deeper grammatical understanding of the text, moving beyond simple word counts. By identifying nouns, verbs, adjectives, etc., we can infer the focus and style of news articles. For instance, a category heavy in nouns might be very factual and report-driven, while one with more verbs could indicate action-oriented narratives. Differences in POS distribution across categories can highlight stylistic variations—e.g., 'opinion' pieces might have more adjectives and adverbs than 'breaking news' alerts. This information can be invaluable for content generation, genre classification, improving search relevance (by prioritizing certain POS types), or even detecting journalistic bias by analyzing the prevalence of subjective vs. objective language patterns.

## MODULE 5: Dependency Parsing

Dependency parsing is a process that analyzes the grammatical structure of a sentence and identifies the relationships between words. It represents sentence structure as a tree where words are nodes and directed edges (dependencies) connect a head word to its dependents. This module will leverage spaCy to perform dependency parsing, extract syntactic features, and analyze common relationships.

### 5.1 Dependency Parsing and Feature Extraction

We'll define a function to perform dependency parsing on the cleaned content and extract features like the head word, dependency type, and part of speech of both the head and the dependent. This provides granular insight into sentence structure.

In [13]:
def extract_dependency_features(text):
    """
    Performs dependency parsing and extracts syntactic features using spaCy.

    Args:
        text (str): The input text string.

    Returns:
        list: A list of dictionaries, each representing a dependency relationship
              with 'dependent', 'head', 'dep_type', 'dependent_pos', 'head_pos'.
    """
    if not isinstance(text, str) or not text.strip(): # Handle non-string or empty inputs
        return []

    doc = nlp(text)
    dependencies = []
    for token in doc:
        if token.head != token: # Avoid self-loops (root token)
            dependencies.append({
                'dependent': token.text,
                'head': token.head.text,
                'dep_type': token.dep_,
                'dependent_pos': token.pos_,
                'head_pos': token.head.pos_,
            })
    return dependencies


if not df.empty and 'cleaned_content' in df.columns:
    print("Applying dependency parsing and extracting features. This might take some time for large datasets...")
    # Apply to a sample to avoid extremely long processing times for very large datasets
    # or apply to the full dataset if computational resources allow.
    # For this example, let's take a sample of 1000 rows if the df is large
    sample_df = df.sample(min(1000, len(df)), random_state=42).copy() if len(df) > 1000 else df.copy()
    sample_df['dependency_features'] = sample_df['cleaned_content'].apply(extract_dependency_features)
    print("Dependency parsing complete for the sample (or full) dataset.")
    print("\nFirst 5 rows of sample with dependency features (first few relationships):")
    display(sample_df[['cleaned_content', 'dependency_features']].head().apply(lambda x: x.apply(lambda y: y[:2] if isinstance(y, list) else y)))

    # Store the sample_df for later analysis if it was used
    if len(df) > 1000:
        df_parsed_sample = sample_df
    else:
        df_parsed_sample = df # If no sampling, use the full df

else:
    print("DataFrame is empty or 'cleaned_content' column is missing. Cannot perform dependency parsing.")


DataFrame is empty or 'cleaned_content' column is missing. Cannot perform dependency parsing.


### 5.2 Relationship Analysis

We can analyze the most common dependency types and specific relationships to understand typical sentence constructions in the news articles. This can highlight how subjects relate to verbs, or how adjectives modify nouns.

In [14]:
from itertools import chain


if 'df_parsed_sample' in locals() and not df_parsed_sample.empty and 'dependency_features' in df_parsed_sample.columns:
    print("Analyzing dependency relationships...")
    all_dependencies = list(chain.from_iterable(df_parsed_sample['dependency_features']))

    if all_dependencies:
        dependencies_df = pd.DataFrame(all_dependencies)

        # Most common dependency types
        print("\nTop 10 Most Common Dependency Types:")
        dep_type_counts = dependencies_df['dep_type'].value_counts().head(10)
        display(dep_type_counts)

        plt.figure(figsize=(10, 6))
        sns.barplot(x=dep_type_counts.index, y=dep_type_counts.values, palette='coolwarm')
        plt.title('Top 10 Most Common Dependency Types')
        plt.xlabel('Dependency Type')
        plt.ylabel('Frequency')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(os.path.join(outputs_dir, 'common_dependency_types.png'))
        plt.show()

        # Example: Most common Noun Phrase modifiers (amod - adjectival modifier)
        print("\nTop 10 Adjectival Modifiers (amod): dependent (ADJ) -> head (NOUN)")
        amod_deps = dependencies_df[(dependencies_df['dep_type'] == 'amod') &
                                    (dependencies_df['dependent_pos'] == 'ADJ') &
                                    (dependencies_df['head_pos'] == 'NOUN')]
        if not amod_deps.empty:
            amod_counts = amod_deps.apply(lambda x: f"{x['dependent']} -> {x['head']}", axis=1).value_counts().head(10)
            display(amod_counts)

            plt.figure(figsize=(10, 6))
            sns.barplot(x=amod_counts.values, y=amod_counts.index, palette='viridis')
            plt.title('Top 10 Adjectival Modifiers (ADJ -> NOUN)')
            plt.xlabel('Frequency')
            plt.ylabel('Adjective -> Noun Pair')
            plt.tight_layout()
            plt.savefig(os.path.join(outputs_dir, 'top_amod_dependencies.png'))
            plt.show()
        else:
            print("No 'amod' dependencies found in the sample.")

    else:
        print("No dependency features extracted to analyze.")
else:
    print("Dependency parsed DataFrame sample not found or empty. Cannot analyze relationships.")


Dependency parsed DataFrame sample not found or empty. Cannot analyze relationships.


### Business Insight (Module 5)

Dependency parsing moves beyond individual word analysis to understand the syntactical structure and relationships within sentences. This is incredibly valuable for several applications:

*   **Information Extraction:** By understanding how words relate, we can extract specific facts (e.g., 'who did what to whom') more accurately. For example, identifying `nsubj` (nominal subject) and `dobj` (direct object) dependencies allows for precise subject-verb-object triplet extraction, crucial for building knowledge graphs or question-answering systems.
*   **Sentiment Analysis Refinement:** Understanding dependency relationships can help attribute sentiment more accurately. For instance, an adjective's sentiment can be correctly linked to the noun it modifies, even if they are not adjacent.
*   **Summarization:** Key phrases and clauses often hold central ideas. Dependency parsing helps identify these core components, improving the quality of automated summaries.
*   **Chatbot and Conversational AI:** Bots can better understand user intent and complex queries by analyzing the grammatical structure, leading to more natural and accurate responses.

By analyzing common dependency patterns, businesses can gain insights into the typical linguistic constructions used in news, which can inform content generation, refine search algorithms, and build more sophisticated NLP applications tailored to their domain.

## MODULE 6: Sentiment Analysis

Sentiment analysis (or opinion mining) is the use of natural language processing, text analysis, computational linguistics, and biometrics to systematically identify, extract, quantify, and study affective states and subjective information. In this module, we will use VADER (Valence Aware Dictionary and sEntiment Reasoner) to assess the sentiment of our news articles. VADER is a lexicon and rule-based sentiment analysis tool that is specifically attuned to sentiments expressed in social media, and works well on texts from various domains.

### 6.1 Sentiment Analysis using VADER

We will initialize the VADER sentiment intensity analyzer and apply it to our `cleaned_content` to get compound sentiment scores.

In [24]:
analyzer = SentimentIntensityAnalyzer()

def get_vader_sentiment(text):
    """
    Calculates the VADER sentiment scores for a given text.

    Args:
        text (str): The input text string.

    Returns:
        dict: A dictionary containing 'neg', 'neu', 'pos', and 'compound' scores.
    """
    if not isinstance(text, str) or not text.strip():
        return {'neg': 0.0, 'neu': 0.0, 'pos': 0.0, 'compound': 0.0}
    return analyzer.polarity_scores(text)

def classify_sentiment(score):
    """
    Classifies sentiment based on the compound score.
    """
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'


if not df.empty and 'cleaned_content' in df.columns:
    print("Applying VADER sentiment analysis...")
    df['sentiment_scores'] = df['cleaned_content'].apply(get_vader_sentiment)

    # Extract compound score and classify sentiment
    df['compound_score'] = df['sentiment_scores'].apply(lambda x: x['compound'])
    df['sentiment'] = df['compound_score'].apply(classify_sentiment)

    print("Sentiment analysis complete.")
    print("\nFirst 5 rows with sentiment scores and classification:")
    display(df[['content', 'compound_score', 'sentiment']].head())
else:
    print("DataFrame is empty or 'cleaned_content' column is missing. Cannot perform sentiment analysis.")

LookupError: 
**********************************************************************
  Resource [93mvader_lexicon[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('vader_lexicon')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93msentiment/vader_lexicon.zip/vader_lexicon/vader_lexicon.txt[0m

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
    - ''
**********************************************************************


### 6.2 Sentiment Distribution Plots

Visualizing the overall sentiment distribution and the distribution of compound scores provides a clear picture of the general emotional tone of the news corpus.

In [16]:
if not df.empty and 'sentiment' in df.columns and 'compound_score' in df.columns:
    print("Generating sentiment distribution plots...")

    # Plot 1: Overall Sentiment Distribution
    plt.figure(figsize=(8, 6))
    sns.countplot(x='sentiment', data=df, palette='coolwarm', order=['Positive', 'Neutral', 'Negative'])
    plt.title('Overall News Article Sentiment Distribution')
    plt.xlabel('Sentiment')
    plt.ylabel('Number of Articles')
    plt.tight_layout()
    plt.savefig(os.path.join(outputs_dir, 'overall_sentiment_distribution.png'))
    plt.show()

    # Plot 2: Distribution of Compound Sentiment Scores
    plt.figure(figsize=(10, 6))
    sns.histplot(df['compound_score'], bins=30, kde=True, color='skyblue')
    plt.title('Distribution of VADER Compound Sentiment Scores')
    plt.xlabel('Compound Sentiment Score (-1 to 1)')
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.savefig(os.path.join(outputs_dir, 'compound_sentiment_histogram.png'))
    plt.show()

    print("Sentiment distribution plots saved.")
else:
    print("DataFrame is empty or sentiment columns are missing. Cannot generate sentiment distribution plots.")


DataFrame is empty or sentiment columns are missing. Cannot generate sentiment distribution plots.


### 6.3 Category Sentiment Comparisons

Comparing sentiment across different news categories can reveal how emotional tone varies by topic, which is a valuable insight for content producers and consumers alike.

In [17]:
if not df.empty and 'sentiment' in df.columns and 'category' in df.columns:
    print("Comparing sentiment across categories...")

    # Calculate average compound score per category
    category_sentiment = df.groupby('category')['compound_score'].mean().sort_values(ascending=False)

    plt.figure(figsize=(12, 7))
    sns.barplot(x=category_sentiment.values, y=category_sentiment.index, palette='RdYlGn')
    plt.title('Average VADER Compound Sentiment Score by Category')
    plt.xlabel('Average Compound Score')
    plt.ylabel('Category')
    plt.tight_layout()
    plt.savefig(os.path.join(outputs_dir, 'category_avg_sentiment.png'))
    plt.show()

    # Plot sentiment distribution per category
    if len(df['category'].unique()) <= 10: # Limit for readability
        g = sns.catplot(x='sentiment', col='category', col_wrap=3, data=df, kind='count', palette='viridis', height=4, aspect=1.2, order=['Positive', 'Neutral', 'Negative'])
        g.set_axis_labels('Sentiment', 'Number of Articles')
        g.set_titles('Category: {col_name}')
        plt.suptitle('Sentiment Distribution Across Categories', y=1.02)
        plt.tight_layout()
        plt.savefig(os.path.join(outputs_dir, 'category_sentiment_distribution.png'))
        plt.show()
    else:
        print("Too many categories to plot individual sentiment distributions for readability. Showing average sentiment only.")

    print("Category sentiment comparisons saved.")
else:
    print("DataFrame is empty or required columns for category sentiment comparison are missing. Cannot perform comparison.")


DataFrame is empty or required columns for category sentiment comparison are missing. Cannot perform comparison.


### Business Insight (Module 6)

Sentiment analysis is a powerful tool for understanding the emotional undertones of news content. For businesses, this translates to:

*   **Brand Monitoring:** Quickly identify if news related to a brand, product, or service is predominantly positive, negative, or neutral, allowing for timely PR responses.
*   **Market Research:** Gauge public sentiment towards industry trends, competitors, or political events that might impact market conditions.
*   **Content Strategy:** Understand which types of news or topics evoke specific sentiments, guiding editorial decisions or content creation for desired emotional impact.
*   **Risk Management:** Detect negative sentiment spikes early, which could signal emerging crises or reputational damage.
*   **Customer Feedback Analysis:** While applied to news here, the same techniques can be used to analyze customer reviews or social media comments to understand satisfaction and pain points.

The comparison of sentiment across categories highlights thematic differences in emotional expression. For instance, 'Politics' might naturally have more negative sentiment due to debates and controversies, while 'Lifestyle' might be more positive. This contextual understanding is vital for accurate interpretation of sentiment scores and informs targeted strategic actions.

## MODULE 7: Multi-class Classification

This module focuses on building and evaluating machine learning models to classify news articles into their respective categories. We will use the TF-IDF vectorized text data as features and the `category` as the target variable. Three popular classification algorithms will be trained and compared: Logistic Regression, Random Forest, and Multinomial Naive Bayes.

### 7.1 Data Preparation for Classification

Before training the models, we need to prepare our data by splitting it into training and testing sets and ensuring our features (TF-IDF matrix) and labels are correctly formatted.

In [18]:
if not df.empty and 'cleaned_content' in df.columns and 'category' in df.columns:
    print("Preparing data for classification...")

    # Re-vectorize if tfidf_vectorizer or tfidf_matrix are not in locals() or df has changed
    if 'tfidf_vectorizer' not in locals() or 'tfidf_matrix' not in locals() or df['cleaned_content'].shape[0] != tfidf_matrix.shape[0]:
        print("Re-running TF-IDF vectorization for classification.")
        tfidf_vectorizer, tfidf_matrix = perform_tfidf_vectorization(df['cleaned_content'])

    X = tfidf_matrix
    y = df['category']

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    print(f"Data split into training (X_train shape: {X_train.shape}, y_train shape: {y_train.shape}) "
          f"and testing (X_test shape: {X_test.shape}, y_test shape: {y_test.shape}) sets.")
else:
    print("DataFrame is empty or required columns ('cleaned_content', 'category') are missing. Cannot prepare data for classification.")


DataFrame is empty or required columns ('cleaned_content', 'category') are missing. Cannot prepare data for classification.


### 7.2 Model Training and Evaluation

We will train three classification models and evaluate their performance using accuracy, classification reports, and confusion matrices. Functions are defined to streamline this process.

In [19]:
def train_and_evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    """
    Trains a given model, makes predictions, and evaluates its performance.

    Args:
        model: The scikit-learn classifier model.
        X_train, y_train: Training data and labels.
        X_test, y_test: Testing data and labels.
        model_name (str): Name of the model for reporting.

    Returns:
        float: Accuracy score of the model.
    """
    print(f"\n--- Training {model_name} ---")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy for {model_name}: {accuracy:.4f}")

    print(f"\nClassification Report for {model_name}:")
    print(classification_report(y_test, y_pred, zero_division=0))

    print(f"\nConfusion Matrix for {model_name}:")
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=model.classes_, yticklabels=model.classes_)
    plt.title(f'Confusion Matrix for {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.savefig(os.path.join(outputs_dir, f'{model_name.lower().replace(" ", "_")}_confusion_matrix.png'))
    plt.show()
    print(f"Confusion matrix for {model_name} saved.")

    return accuracy


if 'X_train' in locals() and not X_train.empty:
    models = {
        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
        'Random Forest': RandomForestClassifier(random_state=42),
        'Multinomial Naive Bayes': MultinomialNB()
    }

    accuracy_scores = {}
    for name, model in models.items():
        accuracy_scores[name] = train_and_evaluate_model(model, X_train, y_train, X_test, y_test, name)

    print("\n--- Model Comparison ---")
    accuracy_df = pd.DataFrame(accuracy_scores.items(), columns=['Model', 'Accuracy'])
    accuracy_df = accuracy_df.sort_values(by='Accuracy', ascending=False)
    print("\nAccuracy Scores of All Models:")
    display(accuracy_df)

    best_model_name = accuracy_df.iloc[0]['Model']
    best_accuracy = accuracy_df.iloc[0]['Accuracy']
    print(f"\nBest performing model: {best_model_name} with an accuracy of {best_accuracy:.4f}")

    # Save the best model for potential future use (optional)
    # import joblib
    # joblib.dump(models[best_model_name], os.path.join(outputs_dir, f'{best_model_name.lower().replace(" ", "_")}_model.joblib'))
    # print(f"Best model ({best_model_name}) saved to {outputs_dir}.")

else:
    print("Training and testing data not found. Please ensure data preparation was successful.")


Training and testing data not found. Please ensure data preparation was successful.


### Business Insight (Module 7)

Multi-class classification is the cornerstone of automated content organization. For a news intelligence system, this translates directly to:

*   **Automated Categorization:** Efficiently tag incoming news articles with relevant categories (e.g., 'Politics', 'Sports', 'Technology'), saving significant manual effort and enabling rapid content delivery.
*   **Improved Information Retrieval:** Users can filter and search for news much more effectively when articles are accurately categorized.
*   **Content Personalization:** Deliver tailored news feeds to users based on their expressed interests in specific categories.
*   **Content Moderation and Routing:** Automatically direct articles to specific teams or departments based on their topic, or flag content that falls into sensitive categories.

The comparison of different models provides valuable insights into the trade-offs between algorithm complexity and performance. Identifying the 'best' model (e.g., based on accuracy or F1-score) ensures that the system provides the most reliable categorization. Understanding the confusion matrix helps identify specific categories that are frequently misclassified, allowing for further model refinement or feature engineering tailored to those challenging distinctions.

## MODULE 8: Named Entity Recognition (NER)

Named Entity Recognition (NER) is a subtask of information extraction that seeks to locate and classify named entities in text into pre-defined categories such as person names, organizations, locations, monetary values, expressions of times, etc. In this module, we will use spaCy to perform NER on our news content, extract specific entity types, analyze their frequencies, and visualize their distribution across categories.

### 8.1 Named Entity Extraction

We will define a function to extract specific named entities (PERSON, ORG, GPE, DATE, MONEY) from the `cleaned_content` using the spaCy model. These entities provide critical information about 'who', 'what', 'where', 'when', and 'how much' in the news.

In [20]:
def extract_named_entities(text, entity_types=['PERSON', 'ORG', 'GPE', 'DATE', 'MONEY']):
    """
    Extracts specific named entities from text using spaCy.

    Args:
        text (str): The input text string.
        entity_types (list): A list of spaCy entity labels to extract.

    Returns:
        list: A list of dictionaries, each containing 'entity_text' and 'entity_label'.
    """
    if not isinstance(text, str) or not text.strip(): # Handle non-string or empty inputs
        return []

    doc = nlp(text)
    entities = []
    for ent in doc.ents:
        if ent.label_ in entity_types:
            entities.append({'entity_text': ent.text, 'entity_label': ent.label_})
    return entities


if not df.empty and 'cleaned_content' in df.columns:
    print("Extracting named entities from the cleaned content...")
    # Apply to a sample if the dataset is very large to manage processing time
    # Using the same sampling logic as for dependency parsing
    sample_df_ner = df.sample(min(1000, len(df)), random_state=42).copy() if len(df) > 1000 else df.copy()

    sample_df_ner['named_entities'] = sample_df_ner['cleaned_content'].apply(extract_named_entities)
    print("Named entity extraction complete for the sample (or full) dataset.")
    print("\nFirst 5 rows of sample with named entities (first few):")
    display(sample_df_ner[['cleaned_content', 'named_entities']].head().apply(lambda x: x.apply(lambda y: y[:3] if isinstance(y, list) else y)))

    # Store the sample_df_ner for later analysis if it was used
    if len(df) > 1000:
        df_ner_sample = sample_df_ner
    else:
        df_ner_sample = df # If no sampling, use the full df

else:
    print("DataFrame is empty or 'cleaned_content' column is missing. Cannot perform Named Entity Recognition.")


DataFrame is empty or 'cleaned_content' column is missing. Cannot perform Named Entity Recognition.


### 8.2 Entity Frequency Tables and Visualizations

We will analyze the overall frequency of extracted entities and visualize the most common ones. This helps identify prominent people, organizations, locations, etc., mentioned in the news.

In [21]:
from collections import defaultdict


if 'df_ner_sample' in locals() and not df_ner_sample.empty and 'named_entities' in df_ner_sample.columns:
    print("Analyzing entity frequencies...")
    all_entities = [ent for sublist in df_ner_sample['named_entities'] for ent in sublist]

    if all_entities:
        entities_df = pd.DataFrame(all_entities)

        # Overall entity counts
        overall_entity_counts = entities_df['entity_text'].value_counts().head(20)
        print("\nTop 20 Most Common Named Entities (Overall):")
        display(overall_entity_counts)

        plt.figure(figsize=(12, 8))
        sns.barplot(x=overall_entity_counts.values, y=overall_entity_counts.index, palette='crest')
        plt.title('Top 20 Most Common Named Entities (Overall)')
        plt.xlabel('Frequency')
        plt.ylabel('Entity Text')
        plt.tight_layout()
        plt.savefig(os.path.join(outputs_dir, 'top_overall_entities.png'))
        plt.show()

        # Entity label counts
        entity_label_counts = entities_df['entity_label'].value_counts()
        print("\nDistribution of Entity Labels:")
        display(entity_label_counts)

        plt.figure(figsize=(10, 6))
        sns.barplot(x=entity_label_counts.index, y=entity_label_counts.values, palette='magma')
        plt.title('Distribution of Named Entity Labels')
        plt.xlabel('Entity Label')
        plt.ylabel('Frequency')
        plt.tight_layout()
        plt.savefig(os.path.join(outputs_dir, 'entity_label_distribution.png'))
        plt.show()

    else:
        print("No named entities extracted to analyze.")
else:
    print("Named entity DataFrame sample not found or empty. Cannot analyze entity frequencies.")


Named entity DataFrame sample not found or empty. Cannot analyze entity frequencies.


### 8.3 Cross-Category Entity Analysis

To understand which entities are prominent in specific news categories, we will perform a cross-category analysis. This can highlight domain-specific entities.

In [22]:
if 'df_ner_sample' in locals() and not df_ner_sample.empty and 'named_entities' in df_ner_sample.columns and 'category' in df_ner_sample.columns:
    print("Performing cross-category entity analysis...")

    # Flatten entities and merge with categories
    entity_data = []
    for index, row in df_ner_sample.iterrows():
        for entity in row['named_entities']:
            entity_data.append({'category': row['category'], **entity})

    if entity_data:
        cross_category_entities_df = pd.DataFrame(entity_data)

        # Get a few example categories (e.g., top 3 most frequent ones)
        example_categories = df_ner_sample['category'].value_counts().index.tolist()[:3]

        fig, axes = plt.subplots(len(example_categories), 1, figsize=(12, 6 * len(example_categories)))
        fig.suptitle('Top Named Entities by Category', y=1.02, fontsize=16)

        for i, category in enumerate(example_categories):
            category_entities = cross_category_entities_df[cross_category_entities_df['category'] == category]
            if not category_entities.empty:
                top_entities_in_category = category_entities['entity_text'].value_counts().head(10)
                if not top_entities_in_category.empty:
                    sns.barplot(x=top_entities_in_category.values, y=top_entities_in_category.index, palette='flare', ax=axes[i])
                    axes[i].set_title(f'Top Entities in Category: {category}')
                    axes[i].set_xlabel('Frequency')
                    axes[i].set_ylabel('Entity Text')
                else:
                    axes[i].set_title(f'No entities found for Category: {category}')
            else:
                axes[i].set_title(f'No data found for Category: {category}')

        plt.tight_layout()
        plt.savefig(os.path.join(outputs_dir, 'category_specific_entities.png'))
        plt.show()
        print("Cross-category entity analysis visualizations saved.")

    else:
        print("No entity data available for cross-category analysis.")
else:
    print("Named entity DataFrame sample not found, or required columns missing. Cannot perform cross-category entity analysis.")


Named entity DataFrame sample not found, or required columns missing. Cannot perform cross-category entity analysis.


### Business Insight (Module 8)

Named Entity Recognition is a critical component for structuring unstructured text data, transforming raw articles into actionable insights. Its business applications are extensive:

*   **Information Retrieval & Knowledge Graph Construction:** Automatically identify key players (PERSON), organizations (ORG), and locations (GPE) mentioned in news, enabling the creation of searchable databases or knowledge graphs that link related entities and events.
*   **Competitive Intelligence:** Track mentions of competitors, their products, and key personnel across news sources.
*   **Market Intelligence:** Monitor trends in specific industries by tracking organizations and monetary values (MONEY) associated with investments, mergers, or market movements.
*   **Event Detection:** Identify specific dates (DATE) and locations (GPE) to pinpoint when and where significant events are occurring, crucial for geopolitical analysis or disaster response.
*   **Content Tagging & Recommendation:** Enhance automated content tagging, making news articles more discoverable. For example, a user interested in 'Elon Musk' would be shown articles where 'Elon Musk' is identified as a PERSON entity.
*   **Risk & Compliance:** Quickly detect mentions of sensitive entities or relationships that might require further investigation.

Cross-category entity analysis is particularly powerful as it reveals which entities are most salient within different domains. For example, specific politicians (PERSON) might be prominent in 'Politics' news, while certain companies (ORG) might dominate 'Technology' news. This granular understanding allows for highly targeted information extraction and personalized content delivery.

---

## Executive Summary: ITAI2373 NewsBot Intelligence System

This NewsBot Intelligence System comprehensively processes and analyzes news articles, transforming unstructured text into valuable, actionable insights. Across eight distinct modules, the system demonstrates proficiency in advanced Natural Language Processing (NLP) techniques, designed to meet the rigorous demands of a modern information-driven environment.

**Key Achievements and Capabilities:**

*   **Robust Data Preprocessing (Module 2):** Implemented a multi-stage cleaning pipeline including lowercasing, regex cleaning, tokenization, stopword removal, and lemmatization, ensuring high-quality input for subsequent analyses. Initial data exploration provides foundational context.
*   **Feature Engineering with TF-IDF (Module 3):** Successfully converted textual content into numerical features using TF-IDF vectorization. Visualizations of top terms, both overall and category-specific, effectively highlighted dominant themes and unique vocabulary within different news domains.
*   **Grammatical Structure Analysis with POS Tagging (Module 4):** Utilized spaCy for Part-of-Speech tagging, providing a deeper understanding of sentence composition. Analysis and visualization of POS tag frequencies, including cross-category comparisons, revealed stylistic and structural nuances.
*   **Syntactic Relationship Discovery with Dependency Parsing (Module 5):** Employed spaCy to perform dependency parsing, extracting grammatical relationships between words. This capability is crucial for advanced information extraction, sentiment attribution, and building more sophisticated AI applications.
*   **Sentiment Analysis with VADER (Module 6):** Applied VADER for lexicon-based sentiment analysis, classifying news articles as positive, neutral, or negative. Distribution plots and category-wise comparisons offered insights into the emotional tone prevalent across different news topics, valuable for brand monitoring and market research.
*   **Multi-class Classification for Automated Categorization (Module 7):** Trained and evaluated Logistic Regression, Random Forest, and Multinomial Naive Bayes models to accurately classify news articles into predefined categories. Comprehensive evaluation metrics (accuracy, classification reports, confusion matrices) identified the best-performing model for automated content organization and personalization.
*   **Named Entity Recognition (NER) for Information Extraction (Module 8):** Extracted critical entities such as PERSON, ORG, GPE, DATE, and MONEY using spaCy. Frequency tables and cross-category visualizations demonstrated the system's ability to identify key actors, organizations, locations, and temporal/financial data, essential for building knowledge graphs and enhancing information retrieval.

**Business Impact:**

This NewsBot Intelligence System empowers users—from media analysts and market researchers to journalists and the general public—with **efficiency, accuracy, and deep insights**. It automates laborious news monitoring, reduces information overload, and provides data-driven intelligence for strategic decision-making. The modular design ensures scalability and adaptability, making it a powerful tool for navigating the complexities of modern news consumption and content management.

**Future Enhancements:**

Potential future enhancements could include real-time data ingestion, integration with advanced deep learning models for even greater accuracy, multi-lingual support, and user-configurable dashboards for personalized insight generation. However, in its current state, the system represents a robust and feature-rich NLP solution for news intelligence.